In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [176]:
df = pd.read_excel(r'C:\Users\Phillip\Desktop\Python\Python Notebooks (Mine)\Datasets\Data5_WT2025.xlsx', index_col=0, parse_dates=False)
df.index = pd.to_datetime(df.index, format='%Y%m').to_period('M')
df.drop(columns=['Unnamed: 2', 'Unnamed: 9'], inplace=True)
df['AKREX_excess'] = df['AKREX'] - df['RF']
df = df.rename(columns={'Mkt-RF' : 'Mkt', 'SMALL medBM': 'SMALL_medBM', 'SMALL LoBM': 'SMALL_LoBM','SMALL HiBM': 'SMALL_HiBM', 'SMALL HiBM': 'SMALL_HiBM', 'SMALL LoBM': 'SMALL_LoBM', 'SMALL medBM': 'SMALL_medBM', 'SMALL HiBM': 'SMALL_HiBM', 'BIG LoBM': 'BIG_LoBM', 'BIG medBM': 'BIG_medBM', 'BIG HiBM': 'BIG_HiBM'})

In [134]:
def three_factor_regression(returns, period):
    
    #Define the period for the regression
    returns = returns.loc[period]
    # Define the regression formula
    formula = 'AKREX_excess ~ Mkt + SMB + HML'
    
    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    
    # Get the summary of the regression results
    summary = model.summary()
    return summary, model

three_factor_regression(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))[0].tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0022,0.001,1.578,0.117,-0.001,0.005
Mkt,0.8859,0.033,27.044,0.000,0.821,0.951
SMB,-0.1456,0.056,-2.586,0.011,-0.257,-0.034
HML,-0.1578,0.042,-3.791,0.000,-0.240,-0.076


## Question 1 : 5 - Factor Regression

In [4]:
def factor_regression(returns, period):

    #Define the period for the regression
    returns = returns.loc[period]
    # Define the regression formula
    formula = 'AKREX_excess ~ Mkt + SMB + HML + MOM + LIQ'
    
    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    
    # Get the summary of the regression results
    summary = model.summary()
    return summary, model

In [154]:
question_1_results = factor_regression(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))
question_1_results[1].resid.std()  * 100

np.float64(1.7531492121588326)

In [6]:
question_2a_results = factor_regression(returns = df, period = pd.period_range(start='2009-10', end='2016-10', freq='M'))
question_2b_results = factor_regression(returns = df, period = pd.period_range(start='2016-11', end='2023-12', freq='M'))
question_2a_results[0].tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0039,0.002,2.372,0.020,0.001,0.007
Mkt,0.7347,0.045,16.402,0.000,0.646,0.824
SMB,0.0911,0.075,1.216,0.227,-0.058,0.240
HML,-0.1637,0.082,-2.000,0.049,-0.327,-0.001
MOM,-0.0308,0.055,-0.559,0.578,-0.141,0.079
LIQ,-0.0266,0.059,-0.448,0.656,-0.145,0.092


In [7]:
question_2b_results[0].tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0010,0.002,0.472,0.638,-0.003,0.005
Mkt,0.9840,0.050,19.797,0.000,0.885,1.083
SMB,-0.2361,0.088,-2.697,0.009,-0.410,-0.062
HML,-0.1616,0.057,-2.847,0.006,-0.275,-0.049
MOM,0.0400,0.064,0.628,0.532,-0.087,0.167
LIQ,-0.0605,0.062,-0.968,0.336,-0.185,0.064


## Question 3 : Sharpe & IR 

In [8]:
def Sharpe_Ratio(returns, period):
    #Define period
    returns = returns.loc[period]

    #Calculate excess returns mean and variance

    excess_returns = returns['AKREX_excess'].mean()
    volatility = returns['AKREX_excess'].std()
    
    #Calculate Sharpe Ratio
    
    return (excess_returns / volatility).round(3)


In [9]:
Q1_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))
Q2a_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2009-10', end='2016-10', freq='M'))
Q2b_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2016-11', end='2023-12', freq='M'))
Q1_Sharpe, Q2a_Sharpe, Q2b_Sharpe

(np.float64(0.283), np.float64(0.374), np.float64(0.237))

In [ ]:
Q1_idio_st = question_1_results[1].resid.std(ddof=1)
Q2a_idio_st = question_2a_results[1].resid.std(ddof=1)
Q2b_idio_st = question_2b_results[1].resid.std(ddof=1) 

In [203]:
def Information_Ratio(returns, periods):

    #Run Regression
    results, model = factor_regression(returns = returns, period = periods)

    #Calculate Information Ratio
    idio_st = model.resid.std(ddof=1)
    alpha = model.params['Intercept']

    #Calculate Sharpe Ratio
    Sharpe = returns['AKREX_excess'].loc[periods].mean() / returns['AKREX_excess'].loc[periods].std(ddof=1)

    return Sharpe, alpha / idio_st

Total_IR = Information_Ratio(returns = df, periods = pd.period_range(start='2009-10', end='2023-12', freq='M'))
Oct_2009_2016 = Information_Ratio(returns = df, periods = pd.period_range(start='2009-10', end='2016-10', freq='M'))
Nov_2016_Dec_2023 = Information_Ratio(returns = df, periods = pd.period_range(start='2016-11', end = '2023-12', freq='M'))

In [205]:
#Sharpe Ratio of the Market
period_total = pd.period_range(start='2009-10', end='2023-12', freq='M')
period_1 = pd.period_range(start='2009-10', end='2016-10', freq='M')
period_2 = pd.period_range(start='2016-11', end='2023-12', freq='M')
#Market Sharpe Ratio
market_sharpe_total_period = (df['Mkt'] - df['RF']).loc[period_total].mean() / (df['Mkt'] - df['RF']).loc[period_total].std(ddof=1)
market_sharpe_period_1 = (df['Mkt'] - df['RF']).loc[period_1].mean() / (df['Mkt'] - df['RF']).loc[period_1].std(ddof=1)
market_sharpe_period_2 = (df['Mkt'] - df['RF']).loc[period_2].mean() / (df['Mkt'] - df['RF']).loc[period_2].std(ddof=1)
#Print
print(market_sharpe_total_period, market_sharpe_period_1, market_sharpe_period_2)

0.22440326511153832 0.27842833724867677 0.1860544674780844


In [ ]:
#Maximum Optimal Sharpe
Sharpe_Period_Total = np.sqrt(Total_IR[0] ** 2 + Total_IR[1] ** 2)
Sharpe_Period_1 = np.sqrt(Oct_2009_2016[0] ** 2 + Oct_2009_2016[1] ** 2)
Sharpe_Period_2 = np.sqrt(Nov_2016_Dec_2023[0] ** 2 + Nov_2016_Dec_2023[1] ** 2)
#Sharpe Ratio and IR
print(Oct_2009_2016[1], Nov_2016_Dec_2023[1], Total_IR[1])
print(Sharpe_Period_Total, Sharpe_Period_1, Sharpe_Period_2)



0.2816203612359777 0.054246887608347415 0.11801780870718533
0.30653665880279735 0.4680189744786772 0.2427946106043832


## Henriksson and Merton : Market Timing

In [178]:
df['Indicator'] = np.where(df['Mkt'] > df['RF'], 1, 0)
df['TimingFactor'] = df['Indicator'] * df['Mkt']
df.columns = df.columns.str.replace(' ','_')

In [206]:
def market_timing_regression(returns, period):
    #Define the period for the regression
    returns = returns.loc[period]

    # Define the regression formula
    formula = 'AKREX_excess ~ Mkt + TimingFactor + SMB + HML + MOM + LIQ'

    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    # Get the summary of the regression results
    results = model.summary()

    return model, results

market_timing  = market_timing_regression(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))

In [207]:
market_timing[1]

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           AKREX_excess   R-squared:                       0.825
Model:                            OLS   Adj. R-squared:                  0.819
Method:                 Least Squares   F-statistic:                     129.3
Date:                Sat, 05 Apr 2025   Prob (F-statistic):           1.62e-59
Time:                        21:18:21   Log-Likelihood:                 449.50
No. Observations:                 171   AIC:                            -885.0
Df Residuals:                     164   BIC:                            -863.0
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.0011      0.002      0.495      0.621      -0.003       0.006
Mkt              0.8645      0.063     13.814      0.000       0.741       0.988
TimingFactor     0.0545      0.099      0.551      0.583      -0.141       0.250
SMB             -0.1260      0.059     -2.122      0.035      -0.243      -0.009
HML             -0.1653      0.045     -3.645      0.000      -0.255      -0.076
MOM              0.0162      0.042      0.381      0.704      -0.068       0.100
LIQ             -0.0471      0.044     -1.059      0.291      -0.135       0.041
==============================================================================
Omnibus:                        0.268   Durbin-Watson:                   2.058
Prob(Omnibus):                  0.875   Jarque-Bera (JB):                0.083
Skew:                           0.035   Prob(JB):                        0.959
Kurtosis:                       3.082   Cond. No.                         83.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Style Analysis

In [217]:
def unconstrained_style_analysis(returns, period):

    returns = returns.loc[period]
    # Define the regression formula
    formula = 'AKREX_excess ~ SMALL_HiBM + SMALL_LoBM + SMALL_medBM + BIG_medBM + BIG_HiBM + BIG_LoBM + BOND + INTERNATIONAL'
    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    # Get the summary of the regression results
    results = model.summary()

    return results, model

q5_unconstrained_results, q5_unconstrained_model = unconstrained_style_analysis(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))

q5_unconstrained_df = pd.concat([pd.Series({'R^2': q5_unconstrained_model.rsquared * 100})
                     , q5_unconstrained_model.params * 100], axis = 0)
q5_unconstrained_df = pd.concat([q5_unconstrained_df, q5_unconstrained_model.tvalues], axis = 1)
q5_unconstrained_df.columns = ['Unconstrained', 't-stat']
q5_unconstrained_df

,Unconstrained,t-stat
R^2,84.523608,NaN
Intercept,0.058024,0.405900
SMALL_HiBM,-30.368812,-2.977642
SMALL_LoBM,3.523662,0.530224
SMALL_medBM,21.711340,1.583318
BIG_medBM,31.021525,2.543954
BIG_HiBM,17.954393,2.243333
BIG_LoBM,50.350750,6.531385
BOND,32.318636,4.390606
INTERNATIONAL,-6.074001,-0.986964


In [219]:
def constrained_style_analysis(returns, period):

    returns = returns.loc[period]
    # Define the regression formula (constrained so we set -1 for no intercept)
    formula = 'AKREX_excess ~ SMALL_HiBM + SMALL_LoBM + SMALL_medBM + BIG_medBM + BIG_HiBM + BIG_LoBM + BOND + INTERNATIONAL - 1'
    
    #Define Constraint
    constraint = 'SMALL_HiBM + SMALL_LoBM + BIG_HiBM + BIG_LoBM + BIG_medBM + SMALL_medBM + BOND + INTERNATIONAL = 1'
    # Fit the regression model
    model = smf.glm(formula=formula, data=returns, family=sm.families.Gaussian(sm.families.links.Identity())).fit_constrained(constraint)

    return model, model.summary()

q5_model_constrained, q5_results_constrained = constrained_style_analysis(returns = df, period= pd.period_range(start='2009-10', end='2023-12', freq='M'))

q5_constrained_df = pd.concat([pd.Series({'R^2' : q5_model_constrained.pseudo_rsquared()* 100})
                     , q5_model_constrained.params * 100], axis = 0)
q5_constrained_df = pd.concat([q5_constrained_df, q5_model_constrained.tvalues], axis =1)
q5_constrained_df.columns = ['Constrained', 't-stat']
q5_constrained_df

,Constrained,t-stat
R^2,99.304896,NaN
SMALL_HiBM,-29.320357,-2.833935
SMALL_LoBM,5.034613,0.749678
SMALL_medBM,18.587307,1.339310
BIG_medBM,28.300652,2.299839
BIG_HiBM,15.632212,1.934146
BIG_LoBM,54.318669,7.266788
BOND,14.696632,4.668369
INTERNATIONAL,-7.249728,-1.212445


In [186]:
from scipy.optimize import minimize
    
def quadratic_programming_style_analysis(returns, period, factors):

    returns = returns.loc[period]
    #Initial Guess
    n_factors = len(factors)
    guess_0 = np.ones(n_factors) / n_factors
    factors = np.array(factors)

    #Constraints: Weights must sum to 1
    constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
    #No Lower or Outer Bound on weights
    bounds = [(0, 1)] * n_factors 


    #Objective Function
    def objective(weights, returns, factors):
    
        residuals = returns['AKREX_excess'] - np.dot(returns[factors], weights)

        return np.sum(residuals ** 2)

    result = minimize(objective, guess_0, args=(returns, factors), method='SLSQP', bounds=bounds, constraints=constraints)

    optimal_weights = {str(factors[i]): result.x[i]*100 for i in range(len(result.x))}
    optimal_weights['R^2'] = (1-result.fun) * 100
    optimal_weights = pd.DataFrame(optimal_weights, index = [0]).T
    optimal_weights.columns = ['Quadratic Programming (%)']

    return result,optimal_weights.round(3)

In [187]:
q5_constrained_quadratic = quadratic_programming_style_analysis(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'), factors = ['SMALL_HiBM', 'SMALL_LoBM', 'SMALL_medBM','BIG_HiBM','BIG_medBM','BIG_LoBM', 'BOND', 'INTERNATIONAL'])

In [188]:
q5_constrained_quadratic[1]

,Quadratic Programming (%)
SMALL_HiBM,0.000
SMALL_LoBM,1.216
SMALL_medBM,0.000
BIG_HiBM,1.631
BIG_medBM,20.385
BIG_LoBM,62.790
BOND,13.977
INTERNATIONAL,0.000
R^2,94.883


In [225]:
q5 = pd.concat([q5_constrained_df,  q5_unconstrained_df, q5_constrained_quadratic[1]], axis = 1)
q5.columns = ['Constrained', 't-stat', 'Unconstrained', 't-stat', 'Quadratic Programming']
q5.round(2)

,Constrained,t-stat,Unconstrained,t-stat,Quadratic Programming
R^2,99.30,NaN,84.52,NaN,94.88
SMALL_HiBM,-29.32,-2.83,-30.37,-2.98,0.00
SMALL_LoBM,5.03,0.75,3.52,0.53,1.22
SMALL_medBM,18.59,1.34,21.71,1.58,0.00
BIG_medBM,28.30,2.30,31.02,2.54,20.39
BIG_HiBM,15.63,1.93,17.95,2.24,1.63
BIG_LoBM,54.32,7.27,50.35,6.53,62.79
BOND,14.70,4.67,32.32,4.39,13.98
INTERNATIONAL,-7.25,-1.21,-6.07,-0.99,0.00
Intercept,NaN,NaN,0.06,0.41,NaN
